# 08 — Top Customers & Customer Lifetime Value
Identify highest-value customers by revenue, profit, and order frequency.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Top 15 Customers by Lifetime Revenue ─────────────────────────────────────
top_cust = fetch_df(cur, """
    SELECT c.customer_name, c.segment,
           COUNT(DISTINCT o.order_id)  AS total_orders,
           ROUND(SUM(oi.sales),2)      AS lifetime_revenue,
           ROUND(SUM(oi.profit),2)     AS lifetime_profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100,2) AS margin_pct
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_name, c.segment
    ORDER BY lifetime_revenue DESC LIMIT 15
""")

seg_palette = {"Consumer": PALETTE[0], "Corporate": PALETTE[1],
               "Home Office": PALETTE[2]}
colors = [seg_palette.get(s, PALETTE[3]) for s in top_cust["segment"][::-1]]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(top_cust["customer_name"][::-1],
               top_cust["lifetime_revenue"][::-1],
               color=colors, edgecolor="white")
for bar, margin in zip(bars, top_cust["margin_pct"][::-1]):
    ax.text(bar.get_width() + 200,
            bar.get_y() + bar.get_height()/2,
            f"{fmt_usd(bar.get_width())} ({margin}%)",
            va="center", fontsize=8)

handles = [mpatches.Patch(color=v, label=k) for k, v in seg_palette.items()]
ax.legend(handles=handles, frameon=False, fontsize=9, loc="lower right")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.set_xlabel("Lifetime Revenue (USD)")
ax.set_title("Top 15 Customers by Lifetime Revenue")
plt.tight_layout()
plt.show()


In [ ]:
# ── Pareto: Top 20% customers → % of revenue ─────────────────────────────────
all_cust = fetch_df(cur, """
    SELECT c.customer_id,
           ROUND(SUM(oi.sales),2) AS revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_id ORDER BY revenue DESC
""")

df = all_cust.sort_values("revenue", ascending=False).reset_index(drop=True)
df["cum_rev_pct"]  = df["revenue"].cumsum() / df["revenue"].sum() * 100
df["customer_pct"] = (df.index + 1) / len(df) * 100

top20_rev = df[df["customer_pct"] <= 20]["cum_rev_pct"].max()
print(f"Top 20% of customers → {top20_rev:.1f}% of total revenue (Pareto)")

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(df["customer_pct"], df["cum_rev_pct"],
                alpha=0.12, color=PALETTE[0])
ax.plot(df["customer_pct"], df["cum_rev_pct"],
        color=PALETTE[0], linewidth=2)
ax.axvline(20, color=PALETTE[3], linestyle="--", linewidth=1.5,
           label="Top 20% customers")
ax.axhline(top20_rev, color=PALETTE[2], linestyle="--", linewidth=1.5,
           label=f"{top20_rev:.0f}% of revenue")
ax.annotate(f"  {top20_rev:.0f}%",
            xy=(20, top20_rev), fontsize=11,
            color=PALETTE[3], fontweight="bold")
ax.set_title("Pareto Chart: Cumulative Revenue by Customer %")
ax.set_xlabel("% of Customers")
ax.set_ylabel("Cumulative Revenue %")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
